In [106]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection

In [107]:
# Load banglore home prices into a dataframe
df=pd.read_csv('house_prices.csv')
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price(L)
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [108]:
df.shape

(13320, 9)

In [122]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price(L)'],
      dtype='object')

In [120]:
df['area_type'].unique()

array(['Super built-up  Area', 'Plot  Area', 'Built-up  Area',
       'Carpet  Area'], dtype=object)

In [117]:
df['area_type'].value_counts()

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64

In [116]:
# Drop feature  that are not requiered to build our model.
df1=df.drop(['area_type','society','balcony','availability'],axis='columns')
# df1.head()
df1.shape

(13320, 5)

In [115]:
# Data Cleaning handle NA values
df1.isnull().sum()

location       1
size          16
total_sqft     0
bath          73
price(L)       0
dtype: int64

In [114]:
df2=df1.dropna()
df2.head()

,location,size,total_sqft,bath,price(L)
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


In [28]:
df2.isnull().sum()

location      0
size          0
total_sqft    0
bath          0
price         0
dtype: int64

In [29]:
df2.shape

(13246, 5)

# EDA: Exploratory Data Analysis
Add a new feature (integer) for BHK (Bedrooms, Hall, Kitchen).

In [123]:
df2['bhk']=df2['size'].apply(lambda x: int(x.split(' ')[0]))
df2.bhk.unique()

array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18])

In [99]:
# Explore total_sqft feature
def x_float(x):
    try:
        float(x)
    except:
        return False
    return True    

In [124]:
df2[~df2['total_sqft'].apply(x_float)].head(10)

,location,size,total_sqft,bath,price(L),bhk
30,Yelahanka,4 BHK,2100 - 2850,4.0,186.000,4
122,Hebbal,4 BHK,3067 - 8156,4.0,477.000,4
137,8th Phase JP Nagar,2 BHK,1042 - 1105,2.0,54.005,2
165,Sarjapur,2 BHK,1145 - 1340,2.0,43.490,2
188,KR Puram,2 BHK,1015 - 1540,2.0,56.800,2
410,Kengeri,1 BHK,34.46Sq. Meter,1.0,18.500,1
549,Hennur Road,2 BHK,1195 - 1440,2.0,63.770,2
648,Arekere,9 Bedroom,4125Perch,9.0,265.000,9
661,Yelahanka,2 BHK,1120 - 1145,2.0,48.130,2
672,Bettahalsoor,4 Bedroom,3090 - 5002,4.0,445.000,4


Above shows that total_sqft can be a range (e.g., 2100-2850). For such a case, we can just take the average of the min and max values in the range. There are other cases, such as 34.46 sq. meters, which one can convert to square feet using unit conversion. I am going to just drop such corner cases to keep things simple

In [125]:
def convert_sqft_to_num(x):
    tokens=x.split('-')
    if len(tokens)==2:
        return(float(tokens[0])+float(tokens[1]))/2
    try:
        return float(x)
    except:
        return None

In [128]:
df3=df2.copy()
df3.total_sqft=df3.total_sqft.apply(convert_sqft_to_num)
df3=df3[df3.total_sqft.notnull()]
df3.head(3)

,location,size,total_sqft,bath,price(L),bhk
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3


# For below row, it shows total_sqft as 2475, which is an average of the range 2100-2850

In [130]:
df3.iloc[30]

location      Yelahanka
size              4 BHK
total_sqft       2475.0
bath                4.0
price(L)          186.0
bhk                   4
Name: 30, dtype: object